# DSPy — Programación de prompts en lugar de ingeniería de prompts

DSPy reemplaza el diseño manual de prompts por módulos compilables que
optimizan automáticamente las instrucciones usando ejemplos.

In [ ]:
# pip install dspy-ai
import dspy
import os

# Configurar modelo
lm = dspy.LM(
    model='anthropic/claude-haiku-4-5-20251001',
    api_key=os.getenv('ANTHROPIC_API_KEY'),
    max_tokens=512,
)
dspy.configure(lm=lm)
print('DSPy configurado con Claude Haiku')

## 1. Signatures — definir la tarea con tipos

In [ ]:
# Una Signature define: qué entra → qué sale
# DSPy genera el prompt automáticamente a partir de la definición

class ClasificarSentimiento(dspy.Signature):
    """Clasifica el sentimiento de una reseña de producto."""
    resena: str = dspy.InputField(desc='Texto de la reseña del cliente')
    sentimiento: str = dspy.OutputField(desc='positivo, negativo o neutro')
    confianza: float = dspy.OutputField(desc='Confianza entre 0.0 y 1.0')

class ResumirTexto(dspy.Signature):
    """Resume un texto manteniendo los puntos clave."""
    texto: str = dspy.InputField()
    n_puntos: int = dspy.InputField(desc='Número de puntos clave a extraer')
    resumen: str = dspy.OutputField(desc='Resumen en N puntos numerados')

class ExtraerEntidades(dspy.Signature):
    """Extrae entidades nombradas de un texto."""
    texto: str = dspy.InputField()
    personas: list[str] = dspy.OutputField(desc='Lista de nombres de personas')
    organizaciones: list[str] = dspy.OutputField(desc='Lista de organizaciones')
    lugares: list[str] = dspy.OutputField(desc='Lista de lugares geográficos')

# Módulo Predict — el más simple
clasificador = dspy.Predict(ClasificarSentimiento)

resultado = clasificador(resena='El producto llegó antes de tiempo y funciona perfecto. ¡Muy satisfecho!')
print(f'Sentimiento: {resultado.sentimiento}')
print(f'Confianza: {resultado.confianza}')

## 2. ChainOfThought — razonamiento paso a paso

In [ ]:
class AnalizarContrato(dspy.Signature):
    """Analiza una cláusula contractual e identifica riesgos legales."""
    clausula: str = dspy.InputField(desc='Texto de la cláusula del contrato')
    tipo_contrato: str = dspy.InputField(desc='Tipo: arrendamiento, servicio, compraventa, etc.')
    riesgo_nivel: str = dspy.OutputField(desc='alto, medio o bajo')
    riesgos_identificados: list[str] = dspy.OutputField(desc='Lista de riesgos específicos')
    recomendacion: str = dspy.OutputField(desc='Acción recomendada en 1-2 frases')

# ChainOfThought añade un campo 'reasoning' interno que mejora la calidad
analizador = dspy.ChainOfThought(AnalizarContrato)

CLAUSULA = """
El proveedor podrá subcontratar total o parcialmente los servicios objeto de este contrato
sin necesidad de notificación previa al cliente. La responsabilidad del proveedor quedará
limitada en todo caso al importe abonado en los 3 meses anteriores al incidente.
"""

analisis = analizador(clausula=CLAUSULA, tipo_contrato='servicio')
print(f'Nivel de riesgo: {analisis.riesgo_nivel}')
print(f'Riesgos:')
for r in analisis.riesgos_identificados:
    print(f'  - {r}')
print(f'Recomendación: {analisis.recomendacion}')
if hasattr(analisis, 'reasoning'):
    print(f'\nRazonamiento interno:\n{analisis.reasoning[:200]}...')

## 3. Módulos compuestos — pipelines de DSPy

In [ ]:
class GenerarPreguntasBusqueda(dspy.Signature):
    """Genera preguntas de búsqueda para encontrar información relevante."""
    pregunta_usuario: str = dspy.InputField()
    preguntas_busqueda: list[str] = dspy.OutputField(desc='3-5 preguntas específicas para buscar')

class SintetizarRespuesta(dspy.Signature):
    """Sintetiza una respuesta completa a partir de contexto recuperado."""
    pregunta: str = dspy.InputField()
    contexto: str = dspy.InputField(desc='Información recuperada de la búsqueda')
    respuesta: str = dspy.OutputField(desc='Respuesta completa y bien estructurada')
    fuentes_usadas: list[str] = dspy.OutputField(desc='IDs o títulos de fuentes utilizadas')

class RAGSimple(dspy.Module):
    def __init__(self, base_conocimiento: list[dict]):
        super().__init__()
        self.base_conocimiento = base_conocimiento
        self.generar_preguntas = dspy.Predict(GenerarPreguntasBusqueda)
        self.sintetizar = dspy.ChainOfThought(SintetizarRespuesta)

    def buscar(self, query: str) -> list[dict]:
        query_lower = query.lower()
        relevantes = []
        for doc in self.base_conocimiento:
            texto = (doc.get('titulo', '') + ' ' + doc.get('contenido', '')).lower()
            palabras = query_lower.split()
            coincidencias = sum(1 for p in palabras if p in texto)
            if coincidencias > 0:
                relevantes.append((coincidencias, doc))
        relevantes.sort(reverse=True)
        return [doc for _, doc in relevantes[:3]]

    def forward(self, pregunta: str) -> dspy.Prediction:
        # Paso 1: generar preguntas de búsqueda
        preguntas = self.generar_preguntas(pregunta_usuario=pregunta)

        # Paso 2: recuperar documentos para cada pregunta
        docs_encontrados = set()
        for q in preguntas.preguntas_busqueda[:3]:
            for doc in self.buscar(q):
                docs_encontrados.add(doc['id'])

        contexto = '\n\n'.join(
            f'[{d["id"]}] {d["titulo"]}:\n{d["contenido"]}'
            for d in self.base_conocimiento
            if d['id'] in docs_encontrados
        )

        # Paso 3: sintetizar respuesta
        return self.sintetizar(pregunta=pregunta, contexto=contexto)

# Usar la base de conocimiento de soporte del notebook anterior
BASE_CONOCIMIENTO = [
    {'id': 'kb-001', 'titulo': 'Error de conexión a BD', 'contenido': 'Error DB-500: Verifica POSTGRES_URL. Puerto 5432. Comando: pg_isready -h localhost'},
    {'id': 'kb-002', 'titulo': 'Límite de peticiones', 'contenido': 'Error 429: 1000 peticiones/hora en plan básico. Plan Pro: 10.000/hora.'},
    {'id': 'kb-003', 'titulo': 'Instalación Windows', 'contenido': 'pip install acme-sdk[windows]. Requiere Visual C++ Build Tools.'},
    {'id': 'kb-004', 'titulo': 'JWT inválido', 'contenido': 'Error AUTH-401: Token expirado. Renueva con POST /auth/refresh. Expiran a las 24h.'},
]

rag = RAGSimple(BASE_CONOCIMIENTO)
respuesta = rag(pregunta='¿Cómo soluciono el error 429 y qué plan me permite más peticiones?')
print('Respuesta RAG:')
print(respuesta.respuesta)
print(f'\nFuentes: {respuesta.fuentes_usadas}')

## 4. Compilación con BootstrapFewShot

In [ ]:
from dspy.teleprompt import BootstrapFewShot

# Dataset de entrenamiento
DATASET_ENTRENAMIENTO = [
    dspy.Example(
        resena='Producto de excelente calidad, llegó antes de lo esperado.',
        sentimiento='positivo',
        confianza=0.95
    ).with_inputs('resena'),
    dspy.Example(
        resena='No funciona como se describe. Muy decepcionante.',
        sentimiento='negativo',
        confianza=0.90
    ).with_inputs('resena'),
    dspy.Example(
        resena='Cumple con lo prometido. Nada especial pero tampoco malo.',
        sentimiento='neutro',
        confianza=0.75
    ).with_inputs('resena'),
    dspy.Example(
        resena='Increíble relación calidad-precio. Lo recomiendo sin dudarlo.',
        sentimiento='positivo',
        confianza=0.98
    ).with_inputs('resena'),
    dspy.Example(
        resena='Se rompió a la semana. Pésima calidad de materiales.',
        sentimiento='negativo',
        confianza=0.95
    ).with_inputs('resena'),
]

def metrica_exactitud(ejemplo, prediccion, traza=None):
    return prediccion.sentimiento.lower().strip() == ejemplo.sentimiento.lower()

# Compilar el clasificador con few-shot automático
compilador = BootstrapFewShot(metric=metrica_exactitud, max_bootstrapped_demos=3)
clasificador_base = dspy.Predict(ClasificarSentimiento)

clasificador_compilado = compilador.compile(
    clasificador_base,
    trainset=DATASET_ENTRENAMIENTO
)

# Evaluar
casos_test = [
    'El envío tardó 3 semanas. Inaceptable.',
    'Superó todas mis expectativas. Volveré a comprar.',
    'Está bien para el precio que tiene.',
]

print('Clasificador compilado con BootstrapFewShot:')
for caso in casos_test:
    r_base = clasificador_base(resena=caso)
    r_comp = clasificador_compilado(resena=caso)
    print(f'\n"{caso[:50]}"')
    print(f'  Base:      {r_base.sentimiento} (confianza: {r_base.confianza})')
    print(f'  Compilado: {r_comp.sentimiento} (confianza: {r_comp.confianza})')

## 5. Cuándo usar DSPy vs prompts manuales

In [ ]:
from IPython.display import display, HTML

tabla = """
<table style='border-collapse:collapse;width:100%;font-family:sans-serif;font-size:13px'>
<tr style='background:#4A148C;color:white'>
  <th style='padding:10px'>Situación</th>
  <th style='padding:10px'>Usar DSPy</th>
  <th style='padding:10px'>Prompt manual</th>
</tr>
<tr style='background:#f5f5f5'>
  <td style='padding:8px'>Tienes dataset de ejemplos</td>
  <td style='padding:8px;color:green;font-weight:bold'>✅ DSPy puede optimizar automáticamente</td>
  <td style='padding:8px;color:orange'>⚠️ Debes hacer few-shot manual</td>
</tr>
<tr>
  <td style='padding:8px'>Pipeline de múltiples pasos</td>
  <td style='padding:8px;color:green;font-weight:bold'>✅ Módulos componibles y testeables</td>
  <td style='padding:8px;color:orange'>⚠️ Difícil de mantener</td>
</tr>
<tr style='background:#f5f5f5'>
  <td style='padding:8px'>Prototipo rápido</td>
  <td style='padding:8px;color:orange'>⚠️ Curva de aprendizaje inicial</td>
  <td style='padding:8px;color:green;font-weight:bold'>✅ Más rápido para empezar</td>
</tr>
<tr>
  <td style='padding:8px'>Cambio de modelo</td>
  <td style='padding:8px;color:green;font-weight:bold'>✅ Recompila automáticamente</td>
  <td style='padding:8px;color:red'>❌ Tienes que reescribir prompts</td>
</tr>
<tr style='background:#f5f5f5'>
  <td style='padding:8px'>Control fino del prompt</td>
  <td style='padding:8px;color:orange'>⚠️ DSPy genera el prompt</td>
  <td style='padding:8px;color:green;font-weight:bold'>✅ Control total</td>
</tr>
</table>
"""
display(HTML(tabla))

print('\nRegla práctica: usa DSPy cuando tengas >20 ejemplos etiquetados y un pipeline con 3+ pasos.')